#### OOD-Auswertung -- IMDB als Out-of-Distribution-Referenz

Laedt die bereits gespeicherten Posterior-Samples aller Methoden (kein neues Fitting) und wertet sie zusaetzlich auf IMDB-Texten aus (statt nur auf dem AG-News-Testset). IMDB hat keine gueltigen AG-News-Klassenlabels -- deshalb werden nur die label-unabhaengigen Unsicherheits-Kennzahlen berechnet (Predictive Entropy, Expected Entropy, Mutual Information), keine Accuracy/NLL/Brier fuer OOD. Am Ende: AUROC pro Methode, wie gut MI allein zwischen ID (AG News Test) und OOD (IMDB) unterscheidet -- der eigentliche Nachweis, ob die Unsicherheitsschaetzung praktisch nuetzlich ist.


In [1]:
from pathlib import Path

# ============================================================
# CONFIG
# OOD EVALUATION (IMDB)
# ============================================================

MASTER_DIR = Path(
    "/dss/dsshome1/00/ra58vit2/Masterarbeit"
)

QWEN_REPO = MASTER_DIR / "bayes_sub_inf"

BASELINE_SCRIPT = (
    QWEN_REPO / "experiments" / "ag_news_qwen_lora" / "evaluate_saved_baseline.py"
)

MODEL_PATH = (
    QWEN_REPO / "tests" / "qwen_data" / "helper_scripts" / "data_store"
    / "models" / "qwen2.5_0.5B_float16"
)

RESULTS_ROOT = MASTER_DIR / "method_results"
OOD_RESULT_PATH = RESULTS_ROOT / "ood_imdb_comparison.json"

# ------------------------------------------------------------
# OOD-Daten (IMDB) -- gleiche Tokenisierung wie das AG-News-Testset:
# derselbe Qwen-Tokenizer, gleiche Sequenzlaenge, gleiches Padding.
# SEQ_LEN=128, NICHT 32 -- das AG-News-Testset (data.get("test")) ist
# auf voller Laenge (128) tokenisiert, nur das kleine Posterior-Subset
# wird in den Fitting-Notebooks zusaetzlich auf die letzten 32 Tokens
# gekuerzt. Fuer den ID/OOD-Vergleich muss das Format zum Testset passen.
# ------------------------------------------------------------

SEQ_LEN = 128
N_OOD_EXAMPLES = 2000   # Teilmenge von IMDB (25000 Testbeispiele verfuegbar)

EVAL_BATCH_SIZE = 64

# ------------------------------------------------------------
# Welche gespeicherten Runs (ID: AG News Testset, jetzt zusaetzlich
# OOD: IMDB) ausgewertet werden sollen. Passe run_name an, falls die
# Zelle "AVAILABLE RUNS" unten andere Namen findet.
# MILE zeigt jetzt auf den Multi-Chain-Lauf (K=3, 2048 Beispiele) statt
# der alten Single-Chain-Variante (512) -- run_name wie von
# 02_mile_multichain.ipynb erzeugt (RUN_NAME mit N_CHAINS=3).
# ------------------------------------------------------------

RUNS = [
    ("MILE (Multi-Chain, K=3, 2048)", RESULTS_ROOT / "mile_qwen_agnews",
     "mile_multichain_k3_balanced2048_seq32_prior0.02_warmup250_spc15_thin30_testeval"),
    ("MFVI (2048)", RESULTS_ROOT / "mfvi_qwen_agnews",
     "mfvi_balanced2048_seq32_prior0.02_steps200_samples10_testeval"),
    ("Laplace (2048)", RESULTS_ROOT / "laplace_qwen_agnews",
     "laplace_balanced2048_seq32_prior0.02_fscale1.0_samples10"),
    ("Deep Ensemble (N=3)", RESULTS_ROOT / "deep_ensemble_qwen_agnews",
     "deep_ensemble_n3_seq32"),
]

print("Konfigurierte Runs:", len(RUNS))
for label, result_dir, run_name in RUNS:
    print(" -", label, "->", run_name)


Konfigurierte Runs: 4
 - MILE (Multi-Chain, K=3, 2048) -> mile_multichain_k3_balanced2048_seq32_prior0.02_warmup250_spc15_thin30_testeval
 - MFVI (2048) -> mfvi_balanced2048_seq32_prior0.02_steps200_samples10_testeval
 - Laplace (2048) -> laplace_balanced2048_seq32_prior0.02_fscale1.0_samples10
 - Deep Ensemble (N=3) -> deep_ensemble_n3_seq32


In [2]:
# ============================================================
# IMPORTS AND ENVIRONMENT (selbstheilend -- installiert datasets/
# transformers per `uv add` nach, falls sie fehlen sollten)
# ============================================================

import os
import sys
import json
import runpy
import subprocess

import numpy as np
import jax
import jax.numpy as jnp
from jax import random

os.chdir(QWEN_REPO)

if str(QWEN_REPO) not in sys.path:
    sys.path.insert(0, str(QWEN_REPO))

print("Python:", sys.executable)
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

assert QWEN_REPO.exists()
assert BASELINE_SCRIPT.exists()
assert MODEL_PATH.exists(), f"Modellpfad nicht gefunden: {MODEL_PATH}"

UV_PATH = os.path.expanduser("~/.local/bin/uv")


def uv_add(package_name):
    print(f"uv add {package_name} (in {QWEN_REPO}) ...")
    result = subprocess.run(
        [UV_PATH, "add", package_name],
        cwd=str(QWEN_REPO),
        capture_output=True, text=True,
    )
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError(f"uv add {package_name} fehlgeschlagen.")
    print(f"'{package_name}' hinzugefuegt.")


for _ in range(3):
    try:
        import datasets
        import transformers
        break
    except ModuleNotFoundError as exc:
        print(f"Fehlendes Modul: {exc.name}")
        uv_add(exc.name)

print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)


Python: /dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf/.venv/bin/python3
JAX: 0.5.3
Devices: [CudaDevice(id=0)]
datasets: 2.21.0
transformers: 4.44.2


In [3]:
# ============================================================
# GETEILTES MODUL
# ============================================================

import qwen_posterior_utils as qpu

qpu.patch_subspace_curve_predict()


Patched SubspaceBaseModel._predict: lax.cond bypass for concrete train flags active.


In [4]:
# ============================================================
# LOAD QWEN AG-NEWS BASELINE (env/data + LoRA-Struktur)
# ============================================================

baseline_objects = runpy.run_path(str(BASELINE_SCRIPT))

env = baseline_objects["env"]
params = baseline_objects["params"]
data = baseline_objects["data"]
rng_key = baseline_objects["rng_key"]

print("Baseline geladen. Model:", type(env.s_model))


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps


Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: tmp_files/955nvvay_pretrained_params.npy
Saved object shape: (1,)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (7600, 4)
post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.246214
val_acc: 0.913667
val_ece: 0.009068
val_brier: 0.131597
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.251122
test_acc: 0.914474
test_ece: 0.007036
test_brier: 0.131578
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
Baseline geladen. Model: <class 'subspace_inference.curve_optimizer.subspace_curve.LoRACategorySubspace'>


In [5]:
# ============================================================
# REBUILD_FULL_PARAMS (nur die LoRA-Struktur -- identisch fuer
# alle Methoden, siehe 06_evaluate_all.ipynb)
# ============================================================

theta_map, all_leaves, lora_indices, full_tree_definition, unravel_lora = (
    qpu.vectorize_lora_params(params["params"])
)

rebuild_full_params = qpu.build_rebuild_fn(
    all_leaves, lora_indices, full_tree_definition, unravel_lora,
)

print("theta_map shape:", theta_map.shape)


LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672
All MAP parameters finite: True
theta_map shape: (540672,)


In [6]:
# ============================================================
# ID-DATEN (AG News Testset) -- Referenzformat + spaeter ID-Seite
# des ID-vs-OOD-Vergleichs
# ============================================================

x_id, y_id = data.get("test")

print("ID (AG News Test) Beispiele:", int(y_id.shape[0]))
print("ID input_ids shape:", x_id["input_ids"].shape, x_id["input_ids"].dtype)
print("ID attention_mask shape:", x_id["attention_mask"].shape, x_id["attention_mask"].dtype)


ID (AG News Test) Beispiele: 7600
ID input_ids shape: (7600, 128) int32
ID attention_mask shape: (7600, 128) int32


In [7]:
# ============================================================
# OOD-DATEN (IMDB) LADEN + TOKENISIEREN (gleicher Tokenizer,
# gleiche SEQ_LEN/Padding wie das AG-News-Testset)
# ============================================================

from datasets import load_dataset
from transformers import AutoTokenizer

# Tokenizer direkt von HuggingFace -- der lokale MODEL_PATH-Ordner
# enthaelt nur die custom-konvertierten Modellgewichte (kein
# Standard-HF-Format mit config.json), der Tokenizer selbst ist
# aber unveraendert der originale Qwen-Tokenizer.
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.bos_token

imdb = load_dataset("imdb")
imdb_texts = imdb["test"]["text"][:N_OOD_EXAMPLES]

print("IMDB Beispiele geladen:", len(imdb_texts))

encoded = tokenizer(
    imdb_texts,
    padding="max_length",
    truncation=True,
    max_length=SEQ_LEN,
    return_tensors="np",
)

x_ood = {
    "input_ids": jnp.asarray(encoded["input_ids"]),
    "attention_mask": jnp.asarray(encoded["attention_mask"]),
}

# Dummy-Labels -- werden NIE fuer Accuracy/NLL/Brier verwendet (IMDB hat
# keine gueltigen AG-News-Klassen), nur als Platzhalter fuer die
# gemeinsame Funktionssignatur benoetigt.
y_ood = jnp.zeros((len(imdb_texts),), dtype=jnp.int32)

print("OOD input_ids shape:", x_ood["input_ids"].shape, x_ood["input_ids"].dtype)
print("OOD attention_mask shape:", x_ood["attention_mask"].shape, x_ood["attention_mask"].dtype)


IMDB Beispiele geladen: 2000
OOD input_ids shape: (2000, 128) int32
OOD attention_mask shape: (2000, 128) int32


In [8]:
# ============================================================
# FORMAT-VALIDIERUNG -- x_ood muss strukturell zu x_id passen,
# sonst schlaegt die Auswertung weiter unten mysterioes fehl.
# ============================================================

assert set(x_ood.keys()) == set(x_id.keys()), (
    f"Key-Mismatch: x_ood hat {set(x_ood.keys())}, x_id hat {set(x_id.keys())}"
)
assert x_ood["input_ids"].shape[1] == x_id["input_ids"].shape[1], (
    f"Sequenzlaenge passt nicht: OOD={x_ood['input_ids'].shape[1]}, "
    f"ID={x_id['input_ids'].shape[1]}"
)

print("Format-Check OK -- x_ood ist strukturell kompatibel mit x_id.")
print("ID examples: ", int(y_id.shape[0]))
print("OOD examples:", int(y_ood.shape[0]))


Format-Check OK -- x_ood ist strukturell kompatibel mit x_id.
ID examples:  7600
OOD examples: 2000


In [9]:
# ============================================================
# AVAILABLE RUNS ON DISK (Sanity-Check / Discovery)
# ============================================================

result_dirs = sorted({result_dir for _, result_dir, _ in RUNS}, key=str)

for result_dir in result_dirs:
    print(f"\n{result_dir}:")
    if not result_dir.exists():
        print("  (existiert nicht)")
        continue
    found = sorted(result_dir.glob("*_samples.npy"))
    if not found:
        print("  (keine *_samples.npy gefunden)")
    for path in found:
        print(" -", path.name)



/dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/deep_ensemble_qwen_agnews:
 - deep_ensemble_n3_seq32_samples.npy

/dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/laplace_qwen_agnews:
 - laplace_balanced128_seq32_prior0.02_fscale1.0_samples10_samples.npy
 - laplace_balanced128_seq32_prior1.0_fscale1.0_samples10_samples.npy
 - laplace_balanced2048_seq32_prior0.02_fscale1.0_samples10_samples.npy
 - laplace_balanced512_seq32_prior0.02_fscale1.0_samples10_samples.npy
 - laplace_seed1_balanced2048_seq32_prior0.02_fscale1.0_samples10_samples.npy

/dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/mfvi_qwen_agnews:
 - mfvi_balanced128_seq32_prior0.02_steps200_samples10_samples.npy
 - mfvi_balanced2048_seq32_prior0.02_steps200_samples10_testeval_samples.npy
 - mfvi_balanced512_seq32_prior0.02_steps200_samples10_samples.npy
 - mfvi_balanced512_seq32_prior0.02_steps200_samples10_testeval_samples.npy
 - mfvi_seed1_balanced2048_seq32_prior0.02_steps200_samples10_testeval_samples.npy

In [10]:
# ============================================================
# AUROC (Mann-Whitney-U-Formel, keine sklearn-Abhaengigkeit)
# Hoeherer Score = wird als "eher OOD" interpretiert.
# ============================================================

def compute_auroc(id_scores, ood_scores):
    id_scores = np.asarray(id_scores, dtype=np.float64)
    ood_scores = np.asarray(ood_scores, dtype=np.float64)

    all_scores = np.concatenate([id_scores, ood_scores])
    labels = np.concatenate([
        np.zeros(len(id_scores)), np.ones(len(ood_scores)),
    ])

    order = np.argsort(all_scores)
    ranks = np.empty(len(all_scores), dtype=np.float64)
    ranks[order] = np.arange(1, len(all_scores) + 1)

    n_pos = len(ood_scores)
    n_neg = len(id_scores)
    sum_ranks_pos = ranks[labels == 1].sum()
    u_stat = sum_ranks_pos - n_pos * (n_pos + 1) / 2

    return float(u_stat / (n_pos * n_neg))


In [11]:
# ============================================================
# EVALUATE ALL RUNS -- ID (AG News Test) UND OOD (IMDB).
#
# ID-Wahrscheinlichkeiten werden, falls vorhanden, aus der bereits
# gespeicherten {run_name}_probabilities.npy geladen statt neu
# berechnet -- bei MILE (45 Samples x 7.600 Testbeispiele) allein
# schon ~2h Rechenzeit gespart, die im Fitting-Notebook schon einmal
# investiert wurde.
#
# OOD-Ergebnisse werden pro Methode zwischengespeichert (eine JSON-
# Datei je run_name in ood_cache/) -- ein Absturz mitten in der
# RUNS-Schleife kostet dann nur noch die gerade laufende Methode,
# nicht bereits fertige.
# ============================================================

results = []

rng_key, eval_master_key = random.split(rng_key)

ood_cache_dir = RESULTS_ROOT / "ood_cache"
ood_cache_dir.mkdir(parents=True, exist_ok=True)

for label, result_dir, run_name in RUNS:
    samples_path = Path(result_dir) / f"{run_name}_samples.npy"
    probabilities_path = Path(result_dir) / f"{run_name}_probabilities.npy"
    row_cache_path = ood_cache_dir / f"{run_name}_ood_row.json"

    if not samples_path.exists():
        print(f"UEBERSPRUNGEN (nicht gefunden): {label} -> {samples_path}")
        continue

    if row_cache_path.exists():
        with open(row_cache_path, "r", encoding="utf-8") as f:
            row = json.load(f)
        results.append(row)
        print(f"\n=== {label} ({run_name}) === (aus Zwischenspeicherung geladen)")
        print(f"  ID  Mean MI: {row['id_mean_mi']:.4f}   OOD Mean MI: {row['ood_mean_mi']:.4f}")
        print(f"  AUROC (MI):      {row['auroc_mutual_information']:.4f}")
        print(f"  AUROC (Entropy): {row['auroc_predictive_entropy']:.4f}")
        continue

    print(f"\n=== {label} ({run_name}) ===")
    sample_thetas = np.load(samples_path)
    print("Samples shape:", sample_thetas.shape)

    # ---------- ID: bereits gespeicherte Wahrscheinlichkeiten wiederverwenden ----------
    if probabilities_path.exists():
        id_probs = np.load(probabilities_path)
        id_mean_probs = id_probs.mean(axis=0)
        print(f"ID-Wahrscheinlichkeiten aus dem Fitting-Notebook geladen: {probabilities_path.name}")
    else:
        print("Keine gespeicherten ID-Wahrscheinlichkeiten gefunden -- rechne neu.")
        eval_master_key, id_key = random.split(eval_master_key)
        id_probs, id_mean_probs, id_key = qpu.compute_posterior_predictive_probabilities(
            env=env, rebuild_full_params=rebuild_full_params,
            sample_thetas=sample_thetas, x_eval=x_id, y_eval=y_id,
            rng_key=id_key, eval_batch_size=EVAL_BATCH_SIZE,
        )

    id_metrics = qpu.compute_predictive_metrics(
        sample_probabilities=id_probs, mean_probabilities=id_mean_probs, y_true=y_id,
    )

    # ---------- OOD: IMDB ----------
    eval_master_key, ood_key = random.split(eval_master_key)
    ood_probs, ood_mean_probs, ood_key = qpu.compute_posterior_predictive_probabilities(
        env=env, rebuild_full_params=rebuild_full_params,
        sample_thetas=sample_thetas, x_eval=x_ood, y_eval=y_ood,
        rng_key=ood_key, eval_batch_size=EVAL_BATCH_SIZE,
    )
    ood_metrics = qpu.compute_predictive_metrics(
        sample_probabilities=ood_probs, mean_probabilities=ood_mean_probs, y_true=y_ood,
    )
    # NUR die label-unabhaengigen Groessen sind fuer OOD gueltig:
    # predictive_entropy, expected_entropy, mutual_information.
    # accuracy/lppd/nll/brier aus ood_metrics sind bedeutungslos
    # (y_ood ist reines Dummy) und werden bewusst NICHT verwendet.

    auroc_mi = compute_auroc(
        np.asarray(id_metrics["mutual_information"]),
        np.asarray(ood_metrics["mutual_information"]),
    )
    auroc_entropy = compute_auroc(
        np.asarray(id_metrics["predictive_entropy"]),
        np.asarray(ood_metrics["predictive_entropy"]),
    )

    row = {
        "label": label,
        "run_name": run_name,
        "n_samples": int(sample_thetas.shape[0]),
        "id_accuracy": float(id_metrics["accuracy"]),
        "id_nll": float(id_metrics["posterior_predictive_nll"]),
        "id_mean_mi": float(id_metrics["mutual_information"].mean()),
        "id_mean_entropy": float(id_metrics["predictive_entropy"].mean()),
        "ood_mean_mi": float(ood_metrics["mutual_information"].mean()),
        "ood_mean_entropy": float(ood_metrics["predictive_entropy"].mean()),
        "auroc_mutual_information": auroc_mi,
        "auroc_predictive_entropy": auroc_entropy,
    }
    results.append(row)

    # Zwischenspeichern -- ueberlebt einen Kernel-Absturz / Verbindungsabbruch
    with open(row_cache_path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2)

    print(f"  ID  Mean MI: {row['id_mean_mi']:.4f}   OOD Mean MI: {row['ood_mean_mi']:.4f}")
    print(f"  AUROC (MI):      {auroc_mi:.4f}")
    print(f"  AUROC (Entropy): {auroc_entropy:.4f}")
    print(f"  Zwischengespeichert: {row_cache_path}")



=== MILE (Multi-Chain, K=3, 2048) (mile_multichain_k3_balanced2048_seq32_prior0.02_warmup250_spc15_thin30_testeval) ===
Samples shape: (45, 540672)
ID-Wahrscheinlichkeiten aus dem Fitting-Notebook geladen: mile_multichain_k3_balanced2048_seq32_prior0.02_warmup250_spc15_thin30_testeval_probabilities.npy
Accuracy: 0.9082894325256348
LPPD: -0.4502929449081421
Posterior predictive NLL: 0.4502929449081421
Brier Score: 0.20412513613700867
Mean predictive entropy: 0.846939206123352
Mean expected entropy: 0.501831591129303
Mean mutual information: 0.34510767459869385
Evaluation examples: 2000
Evaluation batch size: 64
Posterior samples: 45
Evaluating posterior sample 1/45
Evaluating posterior sample 2/45
Evaluating posterior sample 3/45
Evaluating posterior sample 4/45
Evaluating posterior sample 5/45
Evaluating posterior sample 6/45
Evaluating posterior sample 7/45
Evaluating posterior sample 8/45
Evaluating posterior sample 9/45
Evaluating posterior sample 10/45
Evaluating posterior sample 

In [12]:
# ============================================================
# COMPARISON TABLE
# ============================================================

columns = [
    ("label", "Methode", 22),
    ("id_accuracy", "ID Acc.", 8),
    ("id_mean_mi", "ID MI", 8),
    ("ood_mean_mi", "OOD MI", 8),
    ("auroc_mutual_information", "AUROC(MI)", 10),
    ("auroc_predictive_entropy", "AUROC(Ent.)", 11),
]

header = "  ".join(name.rjust(width) for _, name, width in columns)
print(header)
print("-" * len(header))

for row in results:
    line_parts = []
    for key, _, width in columns:
        value = row[key]
        text = f"{value:.4f}" if isinstance(value, float) else str(value)
        line_parts.append(text.rjust(width))
    print("  ".join(line_parts))

print()
print("AUROC-Interpretation: 0.5 = MI/Entropie unterscheidet ID/OOD nicht")
print("besser als Zufall; naeher an 1.0 = MI/Entropie erkennt OOD gut.")


               Methode   ID Acc.     ID MI    OOD MI   AUROC(MI)  AUROC(Ent.)
-----------------------------------------------------------------------------
MILE (Multi-Chain, K=3, 2048)    0.9083    0.3451    0.0700      0.0091       0.8929
           MFVI (2048)    0.8896    0.0218    0.0368      0.8813       0.9662
        Laplace (2048)    0.9093    0.0576    0.0870      0.7404       0.9452
   Deep Ensemble (N=3)    0.9150    0.0036    0.0130      0.8954       0.9757

AUROC-Interpretation: 0.5 = MI/Entropie unterscheidet ID/OOD nicht
besser als Zufall; naeher an 1.0 = MI/Entropie erkennt OOD gut.


In [13]:
# ============================================================
# SAVE CONSOLIDATED OOD COMPARISON
# ============================================================

with open(OOD_RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "ood_dataset": "imdb",
        "n_id_examples": int(y_id.shape[0]),
        "n_ood_examples": int(y_ood.shape[0]),
        "seq_len": SEQ_LEN,
        "results": results,
    }, f, indent=2)

print("Gespeichert:", OOD_RESULT_PATH)


Gespeichert: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/ood_imdb_comparison.json
